# Playground Series S6E8: Predicting Smartphone Addiction — Elite Rank Average Ensemble 解説付き写し

- **コンペ**: [Predicting Smartphone Addiction (Playground Series - Season 6, Episode 8)](https://www.kaggle.com/competitions/playground-series-s6e8)
- **元notebook**: [S6E8: Elite Rank Average Ensemble \[0.97092\]](https://www.kaggle.com/code/amanatar/s6e8-elite-rank-average-ensemble-0-97092) by **Aman Atar**
- **スコア**: Public/Best Score **0.97092**（ROC-AUC）
- **手法の概要**: 既に公開されている4つの「エリート」submission（別の参加者が作った高スコアsubmission.csv、いずれもLB 0.9707〜0.9709台）を集め、まず相関行列でモデル間の多様性を確認した上で、各submissionの予測値を「順位（ランク）」に変換してから単純平均する **Rank Average** アンサンブルを行い、LB 0.97092まで押し上げている。

> これは学習目的の解説付き写しです。原著者のコード自体は改変していませんが、出力（実行結果）は含まれていません（未実行）。

## 評価指標

- **タスク**: 各参加者の`gaming_hours`などの行動特徴量から、二値ラベル`addicted_label`（スマートフォン依存かどうか）を予測する二値分類タスク。
- **評価指標**: **ROC-AUC**（ROC曲線下面積）。予測確率がどれだけ正しく「依存者を非依存者より高くスコアリングできているか」という順位の正しさを測る指標で、0.5が完全ランダム、1.0が完全な分離を意味する。
- **なぜこの指標か**: 依存/非依存のクラス比率が偏っている可能性があり、また実運用では「スコアの絶対値」より「どちらがより依存傾向が強いか」という順位付けが重要になる場面が多いため、閾値に依存しないAUCが採用されていると考えられる。
- **この手法とのつながり**: ROC-AUCが順位（ランキング）だけで決まる指標であることを逆手に取り、各モデルの生の予測確率をそのまま平均するのではなく、あえて「順位」に変換してから平均する**Rank Average**を採用している。これにより、モデルごとに予測確率のスケール・キャリブレーションが異なっていても、その差に引っ張られずに順位の一致度だけを純粋に統合できる（GLOSSARY.mdの「ランクベースアンサンブル」も参照）。


### 全体の流れ

このNotebookは5段階のシンプルな構成になっている。ライブラリの読み込み → 4つのelite submissionの読み込み → 相関分析（多様性の確認） → Rank Averaging → 提出ファイル作成、という一直線のパイプラインで、モデル学習は一切行わず「既存の複数の高スコア予測結果を賢く混ぜる」ことだけに特化している。

**What（何をしているか）**: `numpy`・`pandas`・`matplotlib`・`seaborn`・`scipy.stats.rankdata`など、データの読み込み・可視化・順位変換に必要なライブラリを読み込んでいる。`rankdata`は後述のRank Averagingの中核となる関数。

**Why（なぜそうするのか）**: このNotebookはモデル学習を一切行わないため、機械学習ライブラリ（scikit-learn等）は不要で、データ操作と統計処理に絞った軽量な構成になっている。

In [ ]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import rankdata
import os
import glob

sns.set_theme(style="whitegrid", palette="muted")

**What（何をしているか）**: `/kaggle/input/`以下を再帰的に走査して全CSVファイルを列挙し、ファイル名・フォルダ名に`'submission'`を含み`'playground'`を含まないものだけを「elite submission」として抽出、各submissionをDataFrameとして読み込んで`id`列を軸に1つの横長DataFrame（`df_blend`）にまとめている。

**Why（なぜそうするのか）**: このアンサンブル手法では、モデルを新たに学習するのではなく、コミュニティで公開されている複数の高スコアsubmission.csv（他の参加者のノートブックの出力データセットとしてこのノートブックにアタッチされている）をそのまま「入力」として使う。ファイルを動的に探索するコードにしておくことで、どのファイルが実際にアタッチされているかを確認しながら、決め打ちのパスに依存せず柔軟に読み込める。4つ未満しか見つからない場合は警告を出し、想定通りの構成になっているかをその場でチェックしている。

In [ ]:
# Dynamically find all attached csv files
all_csvs = []
for root, dirs, files in os.walk('/kaggle/input/', followlinks=True):
    for f in files:
        if f.endswith('.csv'):
            all_csvs.append(os.path.join(root, f))

print("All CSVs found in /kaggle/input/:")
for c in all_csvs:
    print(" ", c)

paths = {}
for p in all_csvs:
    if 'playground' not in p.lower() and 'submission' in p.lower():
        # Extract unique folder name as model name
        parts = p.split('/')
        model_name = parts[-2] if len(parts) > 1 else parts[-1]
        paths[model_name] = p

print("\nSelected elite submissions:")
for name, path in paths.items():
    print(f" - {name}: {path}")

if len(paths) < 4:
    print("\nWARNING: Expected 4 submissions but found", len(paths))

subs = {}
for name, path in paths.items():
    subs[name] = pd.read_csv(path)

ids = subs[list(subs.keys())[0]]['id']
df_blend = pd.DataFrame({'id': ids})

for name, df in subs.items():
    # Handle if they have different column names or something, but usually it's 'addicted_label'
    if 'addicted_label' in df.columns:
        df_blend[name] = df['addicted_label']
    else:
        df_blend[name] = df.iloc[:, 1]

df_blend.head()

**What（何をしているか）**: 4つのelite submission間の予測値の相関行列をヒートマップとして可視化している（下三角のみ表示）。

**Why（なぜそうするのか）**: アンサンブルの効果は「個々のモデルの強さ」だけでなく「モデル間の予測の多様性（違い）」にも大きく依存する。相関が高すぎる（似すぎている）モデル同士を混ぜても、単体モデルからの伸びしろは小さい。事前に相関行列を確認することで、どのモデルの組み合わせが実際に多様性を持っているかを定量的に把握できる（GLOSSARY.mdの「Hill Climbing」などのアンサンブル手法とも関連する視点）。

In [ ]:
plt.figure(figsize=(8, 6))
corr = df_blend.drop('id', axis=1).corr()

mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".4f", cmap="coolwarm", 
            vmin=0.85, vmax=1.0, square=True, linewidths=.5)
plt.title("Elite Models Correlation Matrix", fontsize=14)
plt.show()

**What（何をしているか）**: `rank_average`関数は、各モデルの予測列を`rankdata`で「1位, 2位, ...」という順位に変換し、それをデータ数で割って0〜1の範囲に正規化した上で、4モデル分の正規化順位を単純平均している。

**Why（なぜそうするのか）**: 評価指標がROC-AUCである以上、モデルの予測値そのものの大小関係（順位）だけが最終スコアに影響し、絶対値のスケールやキャリブレーションのズレは無関係になる。生の確率値をそのまま平均すると、予測値の分布が広い（自信過剰な）モデルが結果を支配してしまう可能性があるが、順位に変換してから平均すれば、4モデルを対等な立場で公平に統合できる。

In [ ]:
def rank_average(df):
    ranks = pd.DataFrame()
    for col in df.columns:
        if col != 'id':
            ranks[col] = rankdata(df[col]) / len(df[col])
    return ranks.mean(axis=1)

df_blend['rank_avg'] = rank_average(df_blend.drop(['id'], axis=1))

**What（何をしているか）**: 元の`id`列とRank Averagingで得た`rank_avg`列を`addicted_label`という提出用の列名に変換し、`submission.csv`として書き出している。

**Why（なぜそうするのか）**: Kaggleのコンペでは、指定されたフォーマット（列名・行順）に厳密に従ったCSVファイルを提出する必要がある。ここでは元のsubmissionと同じ`id`の並びを保ったまま、予測値だけをアンサンブル後の値に差し替えている。

In [ ]:
sub_rank = pd.DataFrame({'id': ids, 'addicted_label': df_blend['rank_avg']})
sub_rank.to_csv('submission.csv', index=False)

print("Submission saved successfully!")
sub_rank.head()

## まとめ

このNotebook自体は非常にシンプルだが、示唆に富む点が2つある。1つは、必ずしも自分でモデルを学習しなくても、公開されている複数の高スコアsubmissionを賢く組み合わせるだけでさらにスコアを伸ばせるという「コミュニティのOOF/submission公開文化」の強さ。もう1つは、評価指標の数学的性質（ROC-AUCは順位のみに依存する）を理解した上で、それに整合したアンサンブル手法（Rank Average）を選んでいる点である。単純平均ではなくランク平均を選ぶという判断そのものが、指標への理解に基づいた設計になっている。

一方で、このNotebookが依拠する4つの元submissionの学習方法・検証方法（fold構成やリークの有無）はブラックボックスであり、GLOSSARY.mdの「Honest OOF」の項目にあるように、fold整合性が確認されていない予測を混ぜるリスクは残る点には注意したい。